# TR-Core: Autonomous Symbolic Induction of Compositional Rules

## Overview

This notebook demonstrates **TR-Core** — a tabula-rasa symbolic cognitive architecture that autonomously discovers compositional rules from raw interaction, without pretrained weights, embeddings, or LLMs.

**Key contribution:** An Apriori-style induction engine with binomial statistical filtering that discovers rare compositional rules (conjunctions of 3–4 literals) and generalizes to structurally held-out contexts.

**Results:**
- Phase 3 (rare compositional, 8 attrs, 4608 contexts): balanced acc **0.78–0.80** vs kNN 0.70, RF 0.66, LogReg 0.20
- Phase 4 (Structural-OOD): held-out triple acc **0.96–1.00** (pair rule recovered 24/24 seeds)
- COMPOSE ablation → chance (0.20), isolating it as causal mechanism

**Paper:** [arXiv] | **Code:** https://github.com/Mardon-programm/tabula-rasa-core | **Zenodo DOI:** 10.5281/zenodo.23619568 | **OSF:** https://osf.io/ruvjg

In [ ]:
# Install TR-Core from GitHub
!pip install -q git+https://github.com/Mardon-programm/tabula-rasa-core.git

import sys
sys.path.insert(0, '/kaggle/working/tabula-rasa-core')

import random
import numpy as np
from collections import Counter

from behavioral.program_v3 import BehavioralProgramV3
from behavioral.program import Params
from environment.attribute_tasks_v4 import (
    AttributeTaskGeneratorV4,
    ATTRIBUTE_DOMAINS_V4,
    V4_ATTRIBUTES,
    ACTIONS_V4,
    oracle_action_v4,
    is_success_v4,
    ContextV4,
)

## 1. Phase 4: Structural-OOD Demo

The agent learns `calm ∧ guard → approach` (pair), then must generalize to the **never-seen** triple `calm ∧ guard ∧ night → approach`.

In [ ]:
# Generate data
gen = AttributeTaskGeneratorV4(seed=42)
sets = gen.generate()
print(f"Train: {len(sets.train)}, Known: {len(sets.known)}, Heldout test: {len(sets.test_heldout)}, Row test: {len(sets.test_row)}")

# Initialize agent
bp = BehavioralProgramV3(
    attributes=list(V4_ATTRIBUTES),
    domains=ATTRIBUTE_DOMAINS_V4,
    actions=ACTIONS_V4,
    params=Params(
        develop_interval=50,
        min_coverage=4,
        validate_accuracy=0.85,
        validate_margin=0.05,
        explore_rate=0.0,
    ),
    seed=42,
    max_arity=4,
)

# Collect experience (uniform random actions)
rng = random.Random(42)
experience = []
for ctx in sets.train:
    action = rng.choice(ACTIONS_V4)
    experience.append((ctx, action, is_success_v4(ctx, action), True))
    bp.learn(ctx, action, is_success_v4(ctx, action))

print(f"Experience collected: {len(experience)} steps")
print(f"Rules before induction: {len(bp.rules)}")

# Run Apriori induction
n_new = bp.develop()
print(f"Rules after induction: {len(bp.rules)} (materialized {n_new} new rules)")

# Show active rules
print("\n=== ACTIVE Rules ===")
for r in bp.active_rules:
    cond = " ∧ ".join([f"{l.attribute}={l.value}" for l in r.condition.literals])
    prov = ", ".join(r.provenance) if r.provenance else "—"
    print(f"  {r.id}: {cond} → {r.action.name}  [{prov}]  (evidence: {r.evidence.n_success}/{r.evidence.n_failure})")

In [ ]:
# Test held-out triple
heldout_ctx = {
    "emotion": "calm",
    "task": "guard",
    "hour": "night",
    "agent": "dog",
    "terrain": "forest",
    "weather": "sunny",
}

pred = bp.choose_action(heldout_ctx)
oracle = oracle_action_v4(heldout_ctx)
print(f"=== Held-Out Triple Test ===")
print(f"  Context: calm ∧ guard ∧ night (never seen in training)")
print(f"  Predicted: {pred.name if pred else 'None'}")
print(f"  Oracle:    {oracle}")
print(f"  Result:    {'✅ SUCCESS' if pred and pred.name == oracle else '❌ FAILED'}")

# Check pair rule recovery
pair_found = False
for r in bp.active_rules:
    if r.action.name == "approach":
        lits = {f"{l.attribute}={l.value}" for l in r.condition.literals}
        if "emotion=calm" in lits and "task=guard" in lits and len(lits) == 2:
            pair_found = True
            break
print(f"  Pair rule recovered: {'✅ YES' if pair_found else '❌ NO'}")

## 2. Ablation: COMPOSE Disabled

Same setup, but COMPOSE disabled → should collapse to chance.

In [ ]:
bp_nocompose = BehavioralProgramV3(
    attributes=list(V4_ATTRIBUTES),
    domains=ATTRIBUTE_DOMAINS_V4,
    actions=ACTIONS_V4,
    params=Params(
        develop_interval=50,
        min_coverage=4,
        validate_accuracy=0.85,
        validate_margin=0.05,
        explore_rate=0.0,
    ),
    seed=42,
    max_arity=4,
    allowed_operations=["CREATE", "MODIFY", "GENERALIZE", "SPECIALIZE", "SPLIT", "DELETE"],  # NO COMPOSE
)

for ctx in sets.train:
    action = rng.choice(ACTIONS_V4)
    bp_nocompose.learn(ctx, action, is_success_v4(ctx, action))

bp_nocompose.develop()

pred_nc = bp_nocompose.choose_action(heldout_ctx)
print(f"=== Ablation: COMPOSE Disabled ===")
print(f"  Predicted: {pred_nc.name if pred_nc else 'None'}")
print(f"  Oracle:    {oracle}")
print(f"  Result:    {'✅' if pred_nc and pred_nc.name == oracle else '❌'}")
print(f"  Active rules: {len(bp_nocompose.active_rules)}")

## 3. Phase 3: Rare Compositional Rules (8 attributes)

Agent discovers 4 rare rules (arity 3–4) whose single literals are individually ambiguous.

In [ ]:
from environment.attribute_tasks_v3 import (
    AttributeTaskGeneratorV3,
    ATTRIBUTE_DOMAINS_V3,
    V3_ATTRIBUTES,
    ACTIONS_V3,
    oracle_action_v3,
    is_success_v3,
)

gen3 = AttributeTaskGeneratorV3(seed=42, train_size=500, known_size=100, t1_size=50, t2_size=50, t3_size=50)
sets3 = gen3.generate()

bp3 = BehavioralProgramV3(
    attributes=list(V3_ATTRIBUTES),
    domains=ATTRIBUTE_DOMAINS_V3,
    actions=ACTIONS_V3,
    params=Params(
        develop_interval=50,
        min_coverage=4,
        validate_accuracy=0.85,
        validate_margin=0.05,
        explore_rate=0.2,
    ),
    seed=42,
    max_arity=4,
)

rng3 = random.Random(42)
for ctx in sets3.train:
    action = rng3.choice(ACTIONS_V3)
    bp3.learn(ctx, action, is_success_v3(ctx, action))

n_new3 = bp3.develop()
print(f"Phase 3: {len(bp3.active_rules)} active rules, {n_new3} materialized")

print("\n=== Rare Compositional Rules (arity ≥ 3) ===")
for r in bp3.active_rules:
    if len(r.condition.literals) >= 3:
        cond = " ∧ ".join([f"{l.attribute}={l.value}" for l in r.condition.literals])
        prov = ", ".join(r.provenance) if r.provenance else "—"
        print(f"  {r.id}: {cond} → {r.action.name}  [{prov}]")

## 4. Formal Complexity Analysis

```
Worst-case time:    O(N·L + L^K)     where N=trials, L=literals, K=max arity
Expected time:      O((1+βL)^K)      β < 1 (coverage decay under pruning)
Space:              O(N + L + R·K)   R = materialized rules
Type I error:       ≤ α = 1e-3       (exact binomial test)
Recovery:           w.h.p. if coverage ≥ c_min, precision ≥ θ+δ, p-value ≤ α/2
```

**Key insight:** Coverage floor c_min=4 induces phase transition: effective branching factor b_eff(k) < 1 for k≥3, making search tractable.

## 5. References

- **Paper**: TR-Core: Autonomous Symbolic Induction of Compositional Rules via Apriori Search with Binomial Filtering (arXiv preprint)
- **Code**: https://github.com/Mardon-programm/tabula-rasa-core
- **Zenodo DOI**: 10.5281/zenodo.23619568
- **OSF Preprint**: https://osf.io/ruvjg
- **Formal Analysis**: `docs/apriori_formal_analysis.tex` in repository

## Citation

```bibtex
@article{trcore2026,
  title={TR-Core: Autonomous Symbolic Induction of Compositional Rules via Apriori Search with Binomial Filtering},
  author={TR-Core Research Group},
  year={2026},
  note={arXiv preprint}
}
```